# Data Lakehouse Tutorial - Part 2: Bronze Layer Data Ingestion

## Overview
This notebook demonstrates how data flows from the OLTP database through Kafka to the Bronze layer in our data lakehouse.

### What is the Bronze Layer?
- **Raw Data**: Stores data exactly as received from source systems
- **Full History**: Preserves all changes including CDC metadata
- **Minimal Processing**: Only basic format conversion (JSON to Parquet)
- **Partition Strategy**: Organized by ingestion time (year/month/day/hour)

### Data Sources
- **PostgreSQL OLTP**: Order management system
- **Debezium CDC**: Captures database changes
- **Kafka**: Streams change events
- **Airflow**: Orchestrates batch ingestion

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import json
from datetime import datetime, timedelta
from minio import Minio

# Initialize Spark session
spark = SparkSession.builder \
    .appName("BronzeLayerExploration") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark session created: {spark.sparkContext.appName}")

## 1. Explore Bronze Layer Structure

Let's examine what data exists in our Bronze layer and understand the partitioning strategy.

In [ ]:
def explore_bronze_structure():
    """
    Explore the directory structure and data in the Bronze layer.
    """
    minio_client = Minio(
        endpoint="minio:9000",
        access_key="minioadmin",
        secret_key="minioadmin123",
        secure=False
    )
    
    print("=== BRONZE LAYER STRUCTURE ===")
    
    try:
        # List objects in bronze bucket
        objects = list(minio_client.list_objects("bronze", recursive=True))
        
        if not objects:
            print("❌ No data found in Bronze layer")
            print("\n🔧 To generate data:")
            print("   1. Run the 'init_and_seed_oltp' DAG in Airflow")
            print("   2. Run the 'stream_new_orders' DAG to generate orders")
            print("   3. Wait for 'kafka_to_bronze' DAG to ingest data")
            return
        
        # Organize objects by table
        tables = {}
        for obj in objects:
            parts = obj.object_name.split('/')
            if len(parts) >= 2:
                table_name = parts[0]
                if table_name not in tables:
                    tables[table_name] = []
                tables[table_name].append(obj)
        
        for table, table_objects in tables.items():
            print(f"\n📁 Table: {table}")
            print(f"   Files: {len(table_objects)}")
            print(f"   Total Size: {sum(obj.size for obj in table_objects):,} bytes")
            
            # Show some example partitions
            partitions = set()
            for obj in table_objects[:10]:  # Show first 10 for brevity
                path_parts = obj.object_name.split('/')
                if len(path_parts) >= 5:  # table/year=../month=../day=../hour=../file
                    partition = '/'.join(path_parts[1:5])
                    partitions.add(partition)
            
            if partitions:
                print(f"   Sample Partitions:")
                for partition in sorted(list(partitions))[:5]:
                    print(f"     - {partition}")
    
    except Exception as e:
        print(f"Error exploring Bronze structure: {e}")

explore_bronze_structure()

## 2. Understanding Debezium CDC Data Format

Bronze layer contains raw CDC events from Debezium. Let's examine the structure:

In [ ]:
def analyze_bronze_data(table_name="orders", limit=3):
    """
    Analyze the structure and content of Bronze layer data.
    """
    try:
        bronze_path = f"s3a://bronze/{table_name}/"
        df = spark.read.format("parquet").load(bronze_path)
        
        print(f"=== BRONZE LAYER ANALYSIS: {table_name.upper()} ===")
        print(f"Total Records: {df.count():,}")
        print(f"\n📋 Schema:")
        df.printSchema()
        
        # Show CDC operation distribution
        print(f"\n📊 CDC Operation Distribution:")
        op_dist = df.groupBy("debezium_op").count().orderBy("count", ascending=False)
        op_dist.show()
        
        # Show partition distribution
        print(f"\n📅 Data Distribution by Day:")
        if "year" in df.columns and "month" in df.columns and "day" in df.columns:
            date_dist = df.groupBy("year", "month", "day").count().orderBy(["year", "month", "day"])
            date_dist.show()
        
        # Show sample CDC events
        print(f"\n🔍 Sample CDC Events:")
        
        # Show different types of operations
        operations = [row['debezium_op'] for row in df.select('debezium_op').distinct().collect()]
        
        for op in operations[:3]:  # Show up to 3 operation types
            print(f"\n--- Operation Type: {op} ---")
            op_df = df.filter(col("debezium_op") == op).limit(1)
            
            # Show key fields for this operation
            key_columns = ["kafka_topic", "kafka_offset", "debezium_op", "debezium_ts_ms", "debezium_source_table"]
            available_key_columns = [c for c in key_columns if c in df.columns]
            
            if available_key_columns:
                op_df.select(available_key_columns).show(1, truncate=False)
            
            # Show business data fields (avoid metadata)
            business_columns = [c for c in df.columns 
                              if not c.startswith(('kafka_', 'debezium_', 'ingestion_'))]
            
            if business_columns:
                print("Business Data:")
                op_df.select(business_columns[:8]).show(1, truncate=False)  # Show first 8 business columns
        
        return df
        
    except Exception as e:
        print(f"❌ Error analyzing {table_name}: {e}")
        print("\n💡 This usually means no data has been ingested yet.")
        return None

# Analyze orders data
orders_df = analyze_bronze_data("orders")

In [ ]:
# Analyze other tables if they exist
for table in ["customers", "products"]:
    print("\n" + "="*60)
    analyze_bronze_data(table, limit=2)

## 3. Understanding CDC Operations

Debezium captures different types of database operations. Let's understand what each means:

In [ ]:
def explain_cdc_operations():
    """
    Explain the different CDC operation types we see in Bronze data.
    """
    operations = {
        'c': {
            'name': 'CREATE (Insert)',
            'description': 'New record was inserted into the database',
            'data_location': 'after field contains the new record data'
        },
        'u': {
            'name': 'UPDATE',
            'description': 'Existing record was modified',
            'data_location': 'before field has old data, after field has new data'
        },
        'd': {
            'name': 'DELETE',
            'description': 'Record was deleted from the database',
            'data_location': 'before field contains the deleted record data'
        },
        'r': {
            'name': 'READ (Snapshot)',
            'description': 'Initial snapshot of existing data during connector startup',
            'data_location': 'after field contains the snapshot data'
        }
    }
    
    print("=== CDC OPERATION TYPES ===")
    for op_code, info in operations.items():
        print(f"\n🔄 '{op_code}' - {info['name']}")
        print(f"   Description: {info['description']}")
        print(f"   Data: {info['data_location']}")
    
    print("\n💡 Key Points:")
    print("   • Bronze layer preserves ALL operations for complete audit trail")
    print("   • Silver layer processes these to maintain current state")
    print("   • Timestamps help with ordering and deduplication")

explain_cdc_operations()

## 4. Data Quality and Completeness Check

Let's examine the quality and completeness of our Bronze layer data:

In [ ]:
def check_bronze_data_quality(df, table_name):
    """
    Perform basic data quality checks on Bronze layer data.
    """
    if df is None:
        return
    
    print(f"\n=== DATA QUALITY CHECK: {table_name.upper()} ===")
    
    total_records = df.count()
    print(f"📊 Total Records: {total_records:,}")
    
    if total_records == 0:
        print("❌ No data to analyze")
        return
    
    # Check for required CDC fields
    required_cdc_fields = [
        "kafka_topic", "kafka_offset", "kafka_partition", 
        "debezium_op", "debezium_ts_ms", "ingestion_timestamp"
    ]
    
    print("\n🔍 CDC Metadata Completeness:")
    for field in required_cdc_fields:
        if field in df.columns:
            null_count = df.filter(col(field).isNull()).count()
            completeness = ((total_records - null_count) / total_records) * 100
            status = "✅" if completeness == 100 else "⚠️"
            print(f"   {status} {field}: {completeness:.1f}% complete")
        else:
            print(f"   ❌ {field}: Missing")
    
    # Check data freshness
    print("\n📅 Data Freshness:")
    if "ingestion_timestamp" in df.columns:
        latest_ingestion = df.agg(max("ingestion_timestamp")).collect()[0][0]
        earliest_ingestion = df.agg(min("ingestion_timestamp")).collect()[0][0]
        
        if latest_ingestion and earliest_ingestion:
            print(f"   🕐 Latest Ingestion: {latest_ingestion}")
            print(f"   🕐 Earliest Ingestion: {earliest_ingestion}")
            
            # Calculate age
            from datetime import datetime
            if isinstance(latest_ingestion, str):
                latest_dt = datetime.fromisoformat(latest_ingestion.replace('Z', '+00:00'))
            else:
                latest_dt = latest_ingestion
            
            age_minutes = (datetime.utcnow().replace(tzinfo=latest_dt.tzinfo) - latest_dt).total_seconds() / 60
            print(f"   ⏰ Data Age: {age_minutes:.1f} minutes")
    
    # Check for duplicates (same kafka topic + partition + offset)
    if all(col_name in df.columns for col_name in ["kafka_topic", "kafka_partition", "kafka_offset"]):
        duplicate_count = df.groupBy("kafka_topic", "kafka_partition", "kafka_offset").count().filter(col("count") > 1).count()
        if duplicate_count > 0:
            print(f"   ⚠️  Found {duplicate_count} duplicate Kafka messages")
        else:
            print(f"   ✅ No duplicate Kafka messages detected")
    
    # Show recent activity
    print("\n📈 Recent Activity (last 10 records by ingestion time):")
    if "ingestion_timestamp" in df.columns:
        recent_columns = ["ingestion_timestamp", "debezium_op", "kafka_offset"]
        available_columns = [c for c in recent_columns if c in df.columns]
        
        if available_columns:
            df.select(available_columns).orderBy(desc("ingestion_timestamp")).show(10, truncate=False)

# Run quality checks on available data
if orders_df:
    check_bronze_data_quality(orders_df, "orders")

## 5. Bronze Layer Querying Patterns

Let's explore common patterns for querying Bronze layer data:

In [ ]:
def demonstrate_bronze_queries(df, table_name):
    """
    Demonstrate common Bronze layer query patterns.
    """
    if df is None or df.count() == 0:
        print(f"No data available for {table_name} queries")
        return
    
    print(f"\n=== BRONZE LAYER QUERY PATTERNS: {table_name.upper()} ===")
    
    # 1. Get latest state for each record (most recent CDC event)
    print("\n1️⃣ Latest State Query (Most Recent CDC Event per Record):")
    
    if table_name == "orders" and "order_id" in df.columns:
        # Window function to get latest record for each order_id
        from pyspark.sql.window import Window
        
        window_spec = Window.partitionBy("order_id").orderBy(desc("debezium_ts_ms"))
        latest_orders = df.withColumn("row_num", row_number().over(window_spec)) \
                         .filter(col("row_num") == 1) \
                         .drop("row_num")
        
        print(f"   Total CDC Events: {df.count():,}")
        print(f"   Latest States: {latest_orders.count():,}")
        
        # Show operation distribution for latest states
        latest_orders.groupBy("debezium_op").count().show()
    
    # 2. Audit trail - show all changes for a specific record
    print("\n2️⃣ Audit Trail Query (All Changes for Specific Records):")
    
    if table_name == "orders" and "order_id" in df.columns:
        # Pick a random order_id that has multiple events
        order_with_changes = df.groupBy("order_id").count().filter(col("count") > 1).limit(1).collect()
        
        if order_with_changes:
            sample_order_id = order_with_changes[0]["order_id"]
            print(f"   📋 Audit trail for Order ID: {sample_order_id}")
            
            audit_columns = ["debezium_ts_ms", "debezium_op", "status", "total_cents"]
            available_audit_columns = [c for c in audit_columns if c in df.columns]
            
            df.filter(col("order_id") == sample_order_id) \
              .select(["order_id"] + available_audit_columns) \
              .orderBy("debezium_ts_ms") \
              .show(truncate=False)
        else:
            print("   No records with multiple changes found")
    
    # 3. Time-based analysis
    print("\n3️⃣ Time-based Analysis (Activity by Hour):")
    
    if "ingestion_timestamp" in df.columns:
        # Group by hour to see ingestion patterns
        hourly_activity = df.withColumn("ingestion_hour", date_format("ingestion_timestamp", "yyyy-MM-dd HH")) \
                           .groupBy("ingestion_hour", "debezium_op") \
                           .count() \
                           .orderBy("ingestion_hour", "debezium_op")
        
        hourly_activity.show(20, truncate=False)
    
    # 4. Data volume analysis
    print("\n4️⃣ Data Volume Analysis (Records per Partition):")
    
    partition_columns = ["year", "month", "day"]
    available_partition_columns = [c for c in partition_columns if c in df.columns]
    
    if available_partition_columns:
        df.groupBy(available_partition_columns).count().orderBy(available_partition_columns).show()

# Demonstrate queries if we have data
if orders_df:
    demonstrate_bronze_queries(orders_df, "orders")

## 6. Understanding Partitioning Strategy

Bronze layer uses time-based partitioning for efficient queries and data management:

In [ ]:
def analyze_partitioning_strategy():
    """
    Analyze and explain the Bronze layer partitioning strategy.
    """
    print("=== BRONZE LAYER PARTITIONING STRATEGY ===")
    
    print("\n📁 Directory Structure:")
    print("   s3a://bronze/")
    print("   ├── orders/")
    print("   │   ├── year=2024/")
    print("   │   │   ├── month=09/")
    print("   │   │   │   ├── day=14/")
    print("   │   │   │   │   ├── hour=10/")
    print("   │   │   │   │   │   └── data_1726315200.parquet")
    print("   │   │   │   │   └── hour=11/")
    print("   │   │   │   └── day=15/")
    print("   │   │   └── month=10/")
    print("   │   └── year=2025/")
    print("   ├── customers/")
    print("   └── products/")
    
    print("\n🎯 Benefits of This Strategy:")
    print("   ✅ Time-based queries are very efficient")
    print("   ✅ Easy to implement data retention policies")
    print("   ✅ Parallel processing of different time periods")
    print("   ✅ Efficient for incremental processing")
    
    print("\n⚡ Query Optimization Examples:")
    print("   # Very fast - uses partition pruning")
    print("   df.filter((col('year') == '2024') & (col('month') == '09'))")
    print("   ")
    print("   # Also fast - single day")
    print("   df.filter((col('year') == '2024') & (col('month') == '09') & (col('day') == '14'))")
    print("   ")
    print("   # Slower - requires full table scan")
    print("   df.filter(col('order_id') == 12345)")
    
    print("\n🕒 Data Retention:")
    print("   • Keep last 90 days in Bronze layer")
    print("   • Archive older data to cold storage")
    print("   • Easy cleanup by dropping old partitions")

analyze_partitioning_strategy()

## 7. Performance Characteristics

Let's understand the performance characteristics of Bronze layer operations:

In [ ]:
def analyze_performance_characteristics(df, table_name):
    """
    Analyze performance characteristics of Bronze layer queries.
    """
    if df is None or df.count() == 0:
        print(f"No data available for performance analysis of {table_name}")
        return
    
    print(f"\n=== PERFORMANCE ANALYSIS: {table_name.upper()} ===")
    
    import time
    
    # 1. Full table scan performance
    print("\n⏱️  Performance Tests:")
    
    start_time = time.time()
    total_count = df.count()
    full_scan_time = time.time() - start_time
    print(f"   📊 Full table count ({total_count:,} records): {full_scan_time:.2f} seconds")
    
    # 2. Partition pruning performance
    if all(col_name in df.columns for col_name in ["year", "month", "day"]):
        start_time = time.time()
        
        # Get the latest partition to test with
        latest_partition = df.select("year", "month", "day").distinct().orderBy(desc("year"), desc("month"), desc("day")).first()
        
        if latest_partition:
            partition_count = df.filter(
                (col("year") == latest_partition["year"]) & 
                (col("month") == latest_partition["month"]) & 
                (col("day") == latest_partition["day"])
            ).count()
            
            partition_scan_time = time.time() - start_time
            print(f"   🎯 Single day partition ({partition_count:,} records): {partition_scan_time:.2f} seconds")
            
            if full_scan_time > 0:
                speedup = full_scan_time / partition_scan_time if partition_scan_time > 0 else float('inf')
                print(f"   🚀 Partition pruning speedup: {speedup:.1f}x faster")
    
    # 3. Memory usage estimation
    print("\n💾 Storage Characteristics:")
    
    # Estimate size per record (rough calculation)
    sample_records = min(1000, df.count())
    if sample_records > 0:
        # This is a rough approximation - actual file sizes would be more accurate
        columns_count = len(df.columns)
        estimated_bytes_per_record = columns_count * 50  # Rough estimate
        estimated_total_mb = (total_count * estimated_bytes_per_record) / (1024 * 1024)
        
        print(f"   📏 Estimated size per record: ~{estimated_bytes_per_record} bytes")
        print(f"   📦 Estimated total data size: ~{estimated_total_mb:.1f} MB")
        print(f"   🗂️  Parquet compression typically reduces this by 70-80%")
    
    # 4. Query patterns analysis
    print("\n🔍 Recommended Query Patterns:")
    print("   ✅ GOOD: Filter by time partitions first")
    print("   ✅ GOOD: Use Spark's adaptive query execution")
    print("   ✅ GOOD: Process data in time-ordered batches")
    print("   ⚠️  AVOID: Full table scans without time filters")
    print("   ⚠️  AVOID: Complex joins on Bronze layer")
    print("   ⚠️  AVOID: Frequent small batch updates")

# Run performance analysis if we have data
if orders_df:
    analyze_performance_characteristics(orders_df, "orders")

## Summary and Next Steps

### What We Learned About Bronze Layer:

1. **Purpose**: Raw data storage with minimal processing
2. **Structure**: Time-based partitioning (year/month/day/hour)
3. **Format**: Parquet files with full CDC metadata
4. **Content**: All database operations (insert, update, delete)
5. **Performance**: Optimized for time-based queries

### Key Takeaways:
- Bronze layer preserves complete audit trail
- Partition pruning dramatically improves query performance
- CDC metadata enables powerful change tracking
- Data quality starts with completeness validation

### Next Steps:
1. **Part 3**: Explore Silver layer transformations
2. **Part 4**: Build Gold layer aggregations
3. **Part 5**: Advanced analytics and monitoring

In [ ]:
# Cleanup
spark.stop()
print("\n✅ Tutorial complete. Spark session stopped.")